# **Remote Training (Colab / GPU)**

This notebook trains the **pitch** and **player** YOLOv8 models on a GPU machine (e.g., Google Colab) and stores all datasets and weights in a stable directory (Google Drive) so you can continue training across sessions and use git for the project code.

In [ ]:
# Environment & paths (designed for Google Colab)
import os

# If running in Colab, optionally mount Google Drive for persistent storage
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
    # Base directory on Drive where data and models will be stored
    BASE_DIR = "/content/drive/MyDrive/football_tracking_remote"
else:
    # Fallback: local path (e.g., when running on your laptop)
    BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

DATA_DIR = os.path.join(BASE_DIR, "datasets")
MODELS_DIR = os.path.join(BASE_DIR, "models")
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

print("IN_COLAB:", IN_COLAB)
print("BASE_DIR:", BASE_DIR)
print("DATA_DIR:", DATA_DIR)
print("MODELS_DIR:", MODELS_DIR)

In [ ]:
# Install dependencies (Ultralytics YOLO + Roboflow + dotenv)
%pip install -q ultralytics roboflow python-dotenv

In [ ]:
# Roboflow setup and common utilities
from roboflow import Roboflow
from dotenv import load_dotenv
import os

# Option A: load ROBOFLOW_API from a .env file in the project root on Drive
env_path = os.path.join(BASE_DIR, "env", "keys.env")
if os.path.exists(env_path):
    load_dotenv(env_path)

# Option B: set it directly here if not using .env
ROBOFLOW_API_KEY = os.getenv("ROBOFLOW_API") or ""  # set manually if empty
if not ROBOFLOW_API_KEY:
    raise RuntimeError("ROBOFLOW_API key not set. Define it in env/keys.env on Drive or directly in this cell.")

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

print("Roboflow client initialized.")

## Pitch detection dataset

In [ ]:
# Download pitch detection dataset (same settings as local training notebook)
project = rf.workspace("roboflow-jvuqo").project("football-field-detection-f07vi")
version = project.version(15)
dataset = version.download("yolov8", location=DATA_DIR)

In [ ]:
# Adjust pitch data.yaml to point to the Roboflow train/val images if needed
# (Most Roboflow YOLOv8 exports already contain correct absolute paths.)
import yaml

pitch_data_yaml = os.path.join(dataset.location, "data.yaml")
print("Pitch data.yaml:", pitch_data_yaml)

with open(pitch_data_yaml, "r") as f:
    data_cfg = yaml.safe_load(f)

print(data_cfg)


## Train pitch detection model (GPU)

This trains a YOLOv8 pose model on the pitch dataset, saving runs under `MODELS_DIR/pitch_detection_model/`.
You can tweak `epochs`, `batch`, and `imgsz` as needed.

In [ ]:
# Train pitch detection model on GPU (if available)
!yolo task=pose mode=train \
    model=yolov8n-pose.pt \
    data={dataset.location}/data.yaml \
    batch=16 epochs=20 imgsz=640 mosaic=0.0 \
    device=0 \
    project={MODELS_DIR} name=pitch_detection_model exist_ok=True

In [ ]:
# Promote pitch model to global best across runs (by mAP50(B))
import os, shutil, json
from datetime import datetime
import pandas as pd

run_dir = os.path.join(MODELS_DIR, "pitch_detection_model")
best_weights = os.path.join(run_dir, "weights", "best.pt")
print("Pitch model (this run):", best_weights)

results_path = os.path.join(run_dir, "results.csv")
global_best_dir = os.path.join(MODELS_DIR, "pitch_detection_model_best")
meta_path = os.path.join(global_best_dir, "meta.json")
metric_name = "metrics/mAP50(B)"

if os.path.exists(results_path) and os.path.exists(best_weights):
    df = pd.read_csv(results_path)
    df.columns = [c.strip() for c in df.columns]
    if metric_name in df.columns:
        current_best = float(df[metric_name].max())
        os.makedirs(global_best_dir, exist_ok=True)

        prev_best = float("-inf")
        if os.path.exists(meta_path):
            try:
                with open(meta_path, "r") as f:
                    prev_best = float(json.load(f).get("best_score", float("-inf")))
            except Exception:
                prev_best = float("-inf")

        if current_best > prev_best:
            target_weights = os.path.join(global_best_dir, "best.pt")
            shutil.copy2(best_weights, target_weights)
            meta = {
                "metric_name": metric_name,
                "best_score": current_best,
                "source_run": run_dir,
                "updated_at": datetime.now().isoformat(),
            }
            with open(meta_path, "w") as f:
                json.dump(meta, f, indent=2)
            print(f"Global best (pitch) updated to {current_best:.4f}, copied to {target_weights}")
        else:
            print(f"Global best (pitch) not updated (current={current_best:.4f}, previous={prev_best:.4f})")
    else:
        print(f"Metric '{metric_name}' not found in results.csv; skipping global-best update for pitch model.")
else:
    print("results.csv or best.pt missing; skipping global-best update for pitch model.")

## Player detection dataset

In [ ]:
# REPLACE this cell's contents with the last 3 lines from your fork's player dataset download snippet
# Example (do NOT run this exact code without updating workspace/project/version):
# project = rf.workspace("your-workspace").project("your-player-project-id")
# version = project.version(20)
# player_dataset = version.download("yolov8", location=DATA_DIR)

raise RuntimeError("Replace this cell with the 3-line Roboflow download snippet for the player dataset, using location=DATA_DIR.")

## Train player detection model (GPU)

This trains a YOLOv8 detection model on the player/ball dataset, saving runs under `MODELS_DIR/player_detection_model/`.

In [ ]:
# Train player detection model on GPU (if available)
!yolo task=detect mode=train \
    model=yolov8x.pt \
    data={player_dataset.location}/data.yaml \
    batch=16 epochs=20 imgsz=1280 \
    device=0 \
    project={MODELS_DIR} name=player_detection_model exist_ok=True

In [ ]:
# Promote player model to global best across runs (by mAP50(B))
import os, shutil, json
from datetime import datetime
import pandas as pd

run_dir = os.path.join(MODELS_DIR, "player_detection_model")
best_weights = os.path.join(run_dir, "weights", "best.pt")
print("Player model (this run):", best_weights)

results_path = os.path.join(run_dir, "results.csv")
global_best_dir = os.path.join(MODELS_DIR, "player_detection_model_best")
meta_path = os.path.join(global_best_dir, "meta.json")
metric_name = "metrics/mAP50(B)"

if os.path.exists(results_path) and os.path.exists(best_weights):
    df = pd.read_csv(results_path)
    df.columns = [c.strip() for c in df.columns]
    if metric_name in df.columns:
        current_best = float(df[metric_name].max())
        os.makedirs(global_best_dir, exist_ok=True)

        prev_best = float("-inf")
        if os.path.exists(meta_path):
            try:
                with open(meta_path, "r") as f:
                    prev_best = float(json.load(f).get("best_score", float("-inf")))
            except Exception:
                prev_best = float("-inf")

        if current_best > prev_best:
            target_weights = os.path.join(global_best_dir, "best.pt")
            shutil.copy2(best_weights, target_weights)
            meta = {
                "metric_name": metric_name,
                "best_score": current_best,
                "source_run": run_dir,
                "updated_at": datetime.now().isoformat(),
            }
            with open(meta_path, "w") as f:
                json.dump(meta, f, indent=2)
            print(f"Global best (player) updated to {current_best:.4f}, copied to {target_weights}")
        else:
            print(f"Global best (player) not updated (current={current_best:.4f}, previous={prev_best:.4f})")
    else:
        print(f"Metric '{metric_name}' not found in results.csv; skipping global-best update for player model.")
else:
    print("results.csv or best.pt missing; skipping global-best update for player model.")